# Train QLoRA Slot Extractor

Use this notebook in Google Colab with a GPU runtime. It trains only the JSON intent/slot extractor adapter; retrieval, dialogue state, validation and booking logic stay deterministic in the main application.

In [ ]:
REPO_URL = "https://github.com/SP-Sauce/AI_Module_CW_Part_B.git"
PROJECT_DIR = "/content/AI_Module_CW_Part_B"

In [2]:
%cd /content
!rm -rf "$PROJECT_DIR"
!git clone "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"

Cloning into '/content/AI_Module_CW_Part_B'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: '/content/AI_Module_CW_Part_B'
/content


In [ ]:
!python -m pip install --upgrade pip
!pip uninstall -y torchao
!pip install -r requirements.txt
!pip install -r requirements-llm.txt
!pip install -r requirements-qlora.txt

In [ ]:
!nvidia-smi
import torch
print("torch.cuda.is_available():", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
!python scripts/check_data_leakage.py
!rm -rf models/slot-extractor-qlora
!rm -f outputs/evaluation/qlora_training_metadata.json
!python scripts/train_qlora_slot_extractor.py \
  --base-model google/flan-t5-small \
  --train-file data/training/slot_instruction_examples.jsonl \
  --eval-file data/evaluation/slot_eval_cases.jsonl \
  --output-dir models/slot-extractor-qlora \
  --metrics-output outputs/evaluation/qlora_training_metadata.json \
  --max-steps 300 \
  --batch-size 2

In [ ]:
!python scripts/run_evaluation_matrix.py \
  --sample-data \
  --slot-fixture data/evaluation/slot_eval_cases.jsonl

In [ ]:
# Optional: preserve the trained adapter and reports in Google Drive.
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
artifact_dir = Path('/content/drive/MyDrive/AI_Module_CW_Part_B_artifacts')
artifact_dir.mkdir(parents=True, exist_ok=True)
for name in ('slot-extractor-qlora', 'evaluation'):
    destination = artifact_dir / name
    if destination.exists():
        shutil.rmtree(destination)
shutil.copytree('models/slot-extractor-qlora', artifact_dir / 'slot-extractor-qlora', dirs_exist_ok=True)
shutil.copytree('outputs/evaluation', artifact_dir / 'evaluation', dirs_exist_ok=True)
print(f'Copied adapter and reports to {artifact_dir}')